# 📊 GitHub Archive — Viral Trend Analysis

This notebook demonstrates:
1. Top 10 viral repositories per time window
2. Language trend comparisons (Python vs TypeScript vs Rust, etc.)
3. Apache Iceberg **time-travel** queries
4. Macro platform statistics

> **Prerequisites**: Run the full pipeline first:
> ```bash
> python main.py --action run-analytics --start-date 2024-01-01 --end-date 2024-01-07
> ```

## 0. Setup SparkSession

In [ ]:
import sys
import os

# Ensure project root is on the path
sys.path.insert(0, os.path.abspath('..'))

from src.utils.spark_session import create_spark_session

# Set use_docker_endpoints=True if running inside the Docker cluster
spark = create_spark_session(
    app_name='Viral_Analysis_Notebook',
    use_docker_endpoints=False,  # Change to True inside Docker
)

print(f'Spark version: {spark.version}')
print('SparkSession ready ✓')

In [ ]:
# Helper: display DataFrames nicely
def show(df, n=20, truncate=False):
    """Pretty-print a Spark DataFrame as a pandas table."""
    return df.limit(n).toPandas()

---
## 1. Top 10 Viral Repositories — Weekly Window

The Virality Index is computed as:
$$\text{VI} = (\text{Stars} \times 4) + (\text{Forks} \times 3) + (\text{PRs opened} \times 2) + (\text{Issues opened} \times 1)$$

In [ ]:
top10_weekly = spark.sql("""
    SELECT
        window_start,
        window_end,
        rank_in_window   AS rank,
        repo_name,
        ROUND(virality_score, 2)  AS virality_score,
        star_count,
        fork_count,
        pr_opened_count,
        issue_opened_count,
        ROUND(star_velocity, 2)   AS stars_per_day,
        active_contributors
    FROM demo.gold.viral_repos
    WHERE window_type = 'week'
      AND rank_in_window <= 10
    ORDER BY window_start DESC, rank_in_window ASC
""")

show(top10_weekly)

### 1b. Top 10 — Daily Window

In [ ]:
top10_daily = spark.sql("""
    SELECT
        window_start           AS date,
        rank_in_window         AS rank,
        repo_name,
        ROUND(virality_score, 2) AS virality_score,
        star_count,
        fork_count
    FROM demo.gold.viral_repos
    WHERE window_type = 'day'
      AND rank_in_window <= 10
    ORDER BY window_start DESC, rank_in_window ASC
""")

show(top10_daily)

### 1c. Visualise — Virality Score Distribution (Weekly Top 10)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

pdf = (
    top10_weekly
    .filter('rank_in_window <= 10')
    .orderBy('window_start', 'rank_in_window')
    .limit(10)
    .toPandas()
)

if not pdf.empty:
    fig, ax = plt.subplots(figsize=(14, 6))
    bars = ax.barh(pdf['repo_name'], pdf['virality_score'], color='steelblue')
    ax.set_xlabel('Virality Score')
    ax.set_title('Top 10 Viral Repos — Weekly Window')
    ax.invert_yaxis()
    ax.bar_label(bars, fmt='%.0f', padding=4)
    plt.tight_layout()
    plt.show()
else:
    print('No data yet — run the pipeline first.')

---
## 2. Language Trend Comparison

Compare Python, TypeScript, Rust, Go, Java and other ecosystems.

In [ ]:
lang_trends = spark.sql("""
    SELECT
        repo_language,
        total_events,
        total_stars,
        total_forks,
        total_prs,
        distinct_repos,
        distinct_contributors,
        ROUND(event_share_pct, 3) AS event_share_pct,
        language_rank
    FROM demo.gold.tech_stack_trends
    ORDER BY language_rank ASC
    LIMIT 20
""")

show(lang_trends, n=20)

In [ ]:
# Filter to specific languages of interest
FOCUS_LANGS = ['Python', 'TypeScript', 'Rust', 'Go', 'Java', 'JavaScript', 'C++', 'Kotlin']

focus_df = spark.sql(f"""
    SELECT repo_language, total_events, total_stars, total_forks,
           distinct_contributors, event_share_pct
    FROM demo.gold.tech_stack_trends
    WHERE repo_language IN ({','.join(repr(l) for l in FOCUS_LANGS)})
    ORDER BY total_events DESC
""").toPandas()

print(focus_df.to_string(index=False))

In [ ]:
import numpy as np

if not focus_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    metrics = [
        ('total_events',         'Total Events',       'cornflowerblue'),
        ('total_stars',          'Stars Received',     'gold'),
        ('distinct_contributors','Active Contributors', 'mediumseagreen'),
    ]

    for ax, (col, title, colour) in zip(axes, metrics):
        bars = ax.bar(focus_df['repo_language'], focus_df[col], color=colour)
        ax.set_title(title)
        ax.set_xlabel('Language')
        ax.tick_params(axis='x', rotation=30)
        ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

    plt.suptitle('Language Ecosystem Comparison', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No language data yet.')

---
## 3. Apache Iceberg Time-Travel Queries

Iceberg stores every snapshot of the table.  
We can query the table **as it was at a previous point in time**.

In [ ]:
# ---- 3a. List all snapshots of the Silver events table ----
snapshots = spark.sql("""
    SELECT snapshot_id, committed_at, operation, summary
    FROM demo.silver.events.snapshots
    ORDER BY committed_at DESC
""")

show(snapshots)

In [ ]:
# ---- 3b. Read the Silver table at a specific snapshot ----
# Replace SNAPSHOT_ID with a real ID from the list above.

snap_rows = snapshots.collect()

if len(snap_rows) >= 2:
    # Use the second-to-last snapshot to illustrate 'before optimization'
    old_snapshot_id = snap_rows[-2]['snapshot_id']
    print(f'Time-travelling to snapshot: {old_snapshot_id}')

    historical_df = spark.read \
        .format('iceberg') \
        .option('snapshot-id', old_snapshot_id) \
        .load('demo.silver.events')

    historical_count = historical_df.count()
    current_count    = spark.table('demo.silver.events').count()

    print(f'Events at snapshot {old_snapshot_id}: {historical_count:,}')
    print(f'Events at current  snapshot:          {current_count:,}')
    print(f'Delta (new events added):             {current_count - historical_count:,}')
elif len(snap_rows) == 1:
    print('Only one snapshot found. Run the pipeline again to see time-travel.')
else:
    print('No snapshots found. Run the pipeline first.')

In [ ]:
# ---- 3c. Time-travel using AS OF TIMESTAMP ----
# Syntax: SELECT ... FROM table TIMESTAMP AS OF '2024-01-02 00:00:00'

if snap_rows:
    ts = snap_rows[-1]['committed_at']   # oldest snapshot
    ts_str = str(ts)[:19]                # YYYY-MM-DD HH:MM:SS

    historical_at_ts = spark.sql(f"""
        SELECT COUNT(*) AS event_count
        FROM demo.silver.events
        TIMESTAMP AS OF '{ts_str}'
    """)
    print(f'Silver events as of {ts_str}:')
    historical_at_ts.show()

In [ ]:
# ---- 3d. Show file-level metadata (before vs after optimization) ----
files_df = spark.sql("""
    SELECT file_path, file_format, record_count, file_size_in_bytes
    FROM demo.silver.events.files
    ORDER BY file_size_in_bytes DESC
    LIMIT 10
""")

show(files_df)

---
## 4. Macro Platform Statistics

In [ ]:
macro = spark.sql("""
    SELECT
        period_start,
        period_end,
        total_events,
        total_stars,
        total_forks,
        total_prs_opened,
        total_prs_merged,
        total_issues_opened,
        total_commits,
        distinct_active_repos,
        distinct_active_contributors,
        ROUND(star_to_fork_ratio, 3)   AS star_to_fork_ratio,
        ROUND(pr_merge_rate, 2)        AS pr_merge_rate_pct,
        ROUND(avg_commits_per_push, 2) AS avg_commits_per_push,
        top_event_type,
        top_language
    FROM demo.gold.macro_stats
    ORDER BY analysis_date DESC
    LIMIT 5
""")

show(macro)

In [ ]:
# Transpose latest macro row for a clean summary card
macro_pdf = macro.limit(1).toPandas()

if not macro_pdf.empty:
    row = macro_pdf.iloc[0]
    print('=' * 45)
    print('       GITHUB PLATFORM MACRO SUMMARY')
    print('=' * 45)
    print(f'  Period              : {row["period_start"]} → {row["period_end"]}')
    print(f'  Total Events        : {row["total_events"]:,}')
    print(f'  Total Stars         : {row["total_stars"]:,}')
    print(f'  Total Forks         : {row["total_forks"]:,}')
    print(f'  PRs Opened          : {row["total_prs_opened"]:,}')
    print(f'  PRs Merged          : {row["total_prs_merged"]:,}')
    print(f'  Issues Opened       : {row["total_issues_opened"]:,}')
    print(f'  Total Commits       : {row["total_commits"]:,}')
    print(f'  Active Repos        : {row["distinct_active_repos"]:,}')
    print(f'  Active Contributors : {row["distinct_active_contributors"]:,}')
    print(f'  Star:Fork Ratio     : {row["star_to_fork_ratio"]}')
    print(f'  PR Merge Rate       : {row["pr_merge_rate_pct"]}%')
    print(f'  Avg Commits/Push    : {row["avg_commits_per_push"]}')
    print(f'  Top Event Type      : {row["top_event_type"]}')
    print(f'  Top Language        : {row["top_language"]}')
    print('=' * 45)
else:
    print('No macro stats yet — run the pipeline first.')

---
## 5. Event Type Distribution Over Time

In [ ]:
event_by_day = spark.sql("""
    SELECT
        event_date,
        event_type,
        COUNT(*) AS event_count
    FROM demo.silver.events
    GROUP BY event_date, event_type
    ORDER BY event_date, event_count DESC
""")

show(event_by_day, n=30)

In [ ]:
import pandas as pd

pdf = event_by_day.toPandas()

if not pdf.empty:
    pivot = pdf.pivot_table(
        index='event_date', columns='event_type', values='event_count', fill_value=0
    )

    ax = pivot.plot(
        kind='area', figsize=(16, 7), alpha=0.7,
        colormap='tab10', stacked=False
    )
    ax.set_title('Daily Event Volume by Type', fontsize=14)
    ax.set_xlabel('Date')
    ax.set_ylabel('Events')
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1), title='Event Type')
    plt.tight_layout()
    plt.show()
else:
    print('No event data yet.')

---
## 6. Silver Table — Schema & Partition Inspection

In [ ]:
# Show table schema
spark.sql('DESCRIBE TABLE EXTENDED demo.silver.events').show(100, truncate=False)

In [ ]:
# Show partition summary
spark.sql("""
    SELECT event_date, COUNT(*) AS records
    FROM demo.silver.events
    GROUP BY event_date
    ORDER BY event_date
""").show(50)

---
## 7. Cleanup

In [ ]:
spark.stop()
print('SparkSession stopped.')